In [1]:
!pip install transformers

In [ ]:
HF_TOKEN="YOUR_HF_TOKEN"

In [3]:
import os

In [4]:
os.environ["HF_TOKEN"]=HF_TOKEN

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [7]:
model_name = "google/gemma-3-1b-it"

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [10]:
tokenizer("Hello World!")

{'input_ids': [2, 9259, 4109, 236888], 'attention_mask': [1, 1, 1, 1]}

In [11]:
input_conversation = [
    {
        "role":"user",
        "content":"Which is the best place to learn GenAI?"
    },
    {
        "role":"assistant",
        "content":"The best place to learn AI is"
    }
]

In [27]:
input_tokens = tokenizer.apply_chat_template(conversation=input_conversation, tokenize=True)
input_tokens

{'input_ids': [2, 105, 2364, 107, 24249, 563, 506, 1791, 1977, 531, 3449, 8471, 12553, 236881, 106, 107, 105, 4368, 107, 818, 1791, 1977, 531, 3449, 12498, 563, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [34]:
input_detokens = tokenizer.apply_chat_template(conversation=input_conversation, tokenize=False, continue_final_message=True)
input_detokens

'<bos><start_of_turn>user\nWhich is the best place to learn GenAI?<end_of_turn>\n<start_of_turn>model\nThe best place to learn AI is'

In [46]:
output_label="GenAI by Lomash Choudhary and use this github repo https://github.com/lomash-choudhary/genAI"
full_conversation = input_detokens + output_label + tokenizer.eos_token
full_conversation

'<bos><start_of_turn>user\nWhich is the best place to learn GenAI?<end_of_turn>\n<start_of_turn>model\nThe best place to learn AI isGenAI by Lomash Choudhary and use this github repo https://github.com/lomash-choudhary/genAI<eos>'

In [47]:
input_tokenized = tokenizer(full_conversation, return_tensors="pt", add_special_tokens=False).to(device)["input_ids"]
input_tokenized

tensor([[     2,    105,   2364,    107,  24249,    563,    506,   1791,   1977,
            531,   3449,   8471,  12553, 236881,    106,    107,    105,   4368,
            107,    818,   1791,   1977,    531,   3449,  12498,    563,  14696,
          12553,    684,  76947,   1316, 162035, 125491,    532,   1161,    672,
          43336,  34691,   4560,   1411,   5846, 236761,    854, 236786,  83908,
           1316, 236772,    574,   3144, 125491, 236786,   2568,  12553,      1]],
       device='cuda:0')

In [48]:
input_ids = input_tokenized[:, :-1].to(device)
target_ids = input_tokenized[:, 1:].to(device)
print("input_ids", input_ids)
print("target_ids", target_ids)

input_ids tensor([[     2,    105,   2364,    107,  24249,    563,    506,   1791,   1977,
            531,   3449,   8471,  12553, 236881,    106,    107,    105,   4368,
            107,    818,   1791,   1977,    531,   3449,  12498,    563,  14696,
          12553,    684,  76947,   1316, 162035, 125491,    532,   1161,    672,
          43336,  34691,   4560,   1411,   5846, 236761,    854, 236786,  83908,
           1316, 236772,    574,   3144, 125491, 236786,   2568,  12553]],
       device='cuda:0')
target_ids tensor([[   105,   2364,    107,  24249,    563,    506,   1791,   1977,    531,
           3449,   8471,  12553, 236881,    106,    107,    105,   4368,    107,
            818,   1791,   1977,    531,   3449,  12498,    563,  14696,  12553,
            684,  76947,   1316, 162035, 125491,    532,   1161,    672,  43336,
          34691,   4560,   1411,   5846, 236761,    854, 236786,  83908,   1316,
         236772,    574,   3144, 125491, 236786,   2568,  12553,      

In [49]:
#function to calculate the loss
import torch.nn as nn
def calculate_loss(logits, labels):
  loss_fn = nn.CrossEntropyLoss(reduction="none")
  cross_entropy = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
  return cross_entropy

In [50]:
import torch

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16).to(device)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [52]:
from torch.optim import AdamW
model.train()

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)

for _ in range(10):
  out = model(input_ids=input_ids)
  loss = calculate_loss(out.logits, target_ids).mean()
  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  print(loss.item())

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.35 GiB is allocated by PyTorch, and 78.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [26]:
# input_prompt="Which is the best place to learn GenAI"
# input_tokens = tokenizer(input_prompt, return_tensors="pt")["input_ids"].to(device)

# output_tokens = model.generate(input_tokens)
# tokenizer.batch_decode(output_tokens)

['<bos>Which is the best place to learn GenAI and Deep Learning?\n\nThere isn\'t one single "best" place, as the ideal choice']

In [45]:
input_prompt=[
    {"role":"user", "content":"Which is the best place to learn GenAI?"}
]

input_ids = tokenizer.apply_chat_template(conversation=input_prompt, tokenize=True, return_tensors="pt").to(device)["input_ids"]

output = model.generate(input_ids, max_new_tokens=25)
print(tokenizer.batch_decode(output, skip_special_tokens=True))

['user\nWhich is the best place to learn GenAI?\nmodel\nThe best place to learn AI isGenAI by Lomash Choudhary']
